<a href="https://colab.research.google.com/github/sydneylaub/rush-sales-analysis/blob/data-cleaning/RUSH_Case_Study_GB885_Final_Project_Laub_S.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RUSH Sportswear — US Sales Analysis

**Analyst:** Sydney Laub

**Prepared for:** VP of US Sales

This notebook analyzes RUSH retail sales data from 2020–2021 to answer four
questions from the VP of US Sales and to surface additional trends relevant
to growth planning.

The source data arrives as three raw tables — sales transactions, retailer
locations, and product definitions — and requires cleaning before analysis.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings for readability
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## Setup and Data Load

The source data lives in three tables:

- **`TABLE_SALES_885.csv`** — one row per order, with units, price, margin, and sales method
- **`TABLE_RETAILER_885.csv`** — retailer locations, one row per retailer-location
- **`TABLE_PRODUCTS_885.csv`** — product category names

All three are read directly from this repository so the notebook runs
without any local setup.

In [ ]:
# Base URL for the raw data files in this repository
BASE_URL = ('https://raw.githubusercontent.com/sydneylaub/'
            'rush-sales-analysis/refs/heads/main/')

# Load each table. Products is pipe-delimited; all files have a UTF-8 BOM.
sales = pd.read_csv(BASE_URL + 'TABLE_SALES_885.csv', encoding='utf-8-sig')
retailers = pd.read_csv(BASE_URL + 'TABLE_RETAILER_885.csv', encoding='utf-8-sig')
products = pd.read_csv(BASE_URL + 'TABLE_PRODUCTS_885.csv',
                       sep='|', encoding='utf-8-sig')

print(f"Sales:     {sales.shape[0]:,} rows × {sales.shape[1]} columns")
print(f"Retailers: {retailers.shape[0]:,} rows × {retailers.shape[1]} columns")
print(f"Products:  {products.shape[0]:,} rows × {products.shape[1]} columns")

## Initial Inspection

Before cleaning, we inspect each table's structure, data types, and value
ranges to identify quality issues. The data is documented as raw and
unvalidated, so we check for type mismatches, missing values, duplicate
keys, inconsistent categories, and outliers.

In [3]:
# Structure and data types of the sales table
sales.info()
sales.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9648 entries, 0 to 9647
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORDER_ID          9648 non-null   int64  
 1   RETAILER_ID       9648 non-null   object 
 2   INVOICE_DATE      9648 non-null   object 
 3   MONTH             9648 non-null   int64  
 4   DAY               9648 non-null   int64  
 5   YEAR              9648 non-null   int64  
 6   PRODUCT_ID        9648 non-null   int64  
 7   PRICE_PER_UNIT    9646 non-null   float64
 8   UNITS_SOLD        9648 non-null   object 
 9   OPERATING_MARGIN  9648 non-null   float64
 10  SALES_METHOD      9648 non-null   object 
dtypes: float64(2), int64(5), object(4)
memory usage: 829.3+ KB


,ORDER_ID,RETAILER_ID,INVOICE_DATE,MONTH,DAY,YEAR,PRODUCT_ID,PRICE_PER_UNIT,UNITS_SOLD,OPERATING_MARGIN,SALES_METHOD
0,1,A00MOHCO,1/1/2020,1,1,2020,20,50.00,1200,0.50,In-store
1,7,A00MOHCO,1/7/2020,1,7,2020,20,50.00,1250,0.50,In-store
2,13,A00MOHCO,1/25/2020,1,25,2020,20,50.00,1220,0.50,Outlet
3,19,A00MOHCO,1/31/2020,1,31,2020,20,50.00,1200,0.50,Outlet
4,25,A00MOHCO,2/6/2020,2,6,2020,20,60.00,1220,0.50,Outlet


In [4]:
# Check for missing values and duplicate orders
print("Missing values by column:")
print(sales.isna().sum()[lambda x: x > 0])
print(f"\nDuplicate ORDER_IDs: {sales['ORDER_ID'].duplicated().sum()}")
print(f"Duplicate full rows:  {sales.duplicated().sum()}")

Missing values by column:
PRICE_PER_UNIT    2
dtype: int64

Duplicate ORDER_IDs: 0
Duplicate full rows:  0


In [5]:
# Inspect categorical fields for inconsistent values
print("Sales methods:")
print(sales['SALES_METHOD'].value_counts())
print("\nOrders by year:")
print(sales['YEAR'].value_counts())

Sales methods:
SALES_METHOD
Online      4889
Outlet      2999
In-store    1740
Ootlet        20
Name: count, dtype: int64

Orders by year:
YEAR
2021    8346
2020    1302
Name: count, dtype: int64


In [6]:
# UNITS_SOLD is stored as text -- identify the non-numeric values
non_numeric = sales[pd.to_numeric(sales['UNITS_SOLD'], errors='coerce').isna()]
print(f"Rows with non-numeric UNITS_SOLD: {len(non_numeric)}")
print(non_numeric[['ORDER_ID', 'RETAILER_ID', 'INVOICE_DATE', 'UNITS_SOLD']])

# Check the numeric ranges for implausible values
print("\nPrice per unit:")
print(sales['PRICE_PER_UNIT'].describe())

Rows with non-numeric UNITS_SOLD: 2
      ORDER_ID RETAILER_ID INVOICE_DATE UNITS_SOLD
1012      6064    S00SALBI    5/27/2021        ***
1439      8626    W00MIODE   12/10/2021        ***

Price per unit:
count    9,646.00
mean        55.58
std      1,017.82
min          7.00
25%         35.00
50%         45.00
75%         55.00
max     99,999.00
Name: PRICE_PER_UNIT, dtype: float64


In [7]:
# The retailer table's primary key should be unique -- verify
print(f"Retailer rows:        {len(retailers)}")
print(f"Unique RETAILER_IDs:  {retailers['RETAILER_ID'].nunique()}")

duplicate_ids = retailers[retailers['RETAILER_ID'].duplicated(keep=False)]
print(f"\nRows with duplicated RETAILER_ID: {len(duplicate_ids)}")
print(duplicate_ids.sort_values('RETAILER_ID'))

# Confirm every sales record maps to a known retailer
orphans = set(sales['RETAILER_ID']) - set(retailers['RETAILER_ID'])
print(f"\nRETAILER_IDs in sales with no match: {orphans}")

Retailer rows:        110
Unique RETAILER_IDs:  106

Rows with duplicated RETAILER_ID: 8
    RETAILER_ID       RETAILER     REGION       STATE         CITY
63     S00NNENE  Sports Direct  Northeast  New Jersey       Newark
64     S00NNENE  Sports Direct  Northeast    New York     New York
81     W00SARLI        Walmart      South    Arkansas  Little Rock
97     W00SARLI      West Gear      South    Arkansas  Little Rock
84     W00SFLOR        Walmart  Southeast     Florida      Orlando
102    W00SFLOR      West Gear  Southeast     Florida      Orlando
83     W00STEHO        Walmart      South       Texas      Houston
100    W00STEHO      West Gear      South       Texas      Houston

RETAILER_IDs in sales with no match: {'999999999'}


In [8]:
# Demonstrate the impact: a naive merge duplicates rows on the collided keys
naive_merge = sales.merge(retailers, on='RETAILER_ID', how='inner')

print(f"Sales rows before merge: {len(sales):,}")
print(f"Rows after naive merge:  {len(naive_merge):,}")
print(f"Rows created by duplicate keys: {len(naive_merge) - len(sales):,}")

Sales rows before merge: 9,648
Rows after naive merge:  10,270
Rows created by duplicate keys: 622


### Summary of Issues Found

| # | Issue | Table | Scope |
|---|-------|-------|-------|
| 1 | `PRODUCT_ID` file is pipe-delimited, not comma | Products | Whole file |
| 2 | UTF-8 byte-order mark corrupts first column name | All three | Whole file |
| 3 | `UNITS_SOLD` stored as text due to `***` placeholder | Sales | 2 rows |
| 4 | `PRICE_PER_UNIT` sentinel value of 99999 | Sales | 1 row |
| 5 | `PRICE_PER_UNIT` missing | Sales | 2 rows |
| 6 | Orders with zero units sold | Sales | 4 rows |
| 7 | `SALES_METHOD` misspelled as "Ootlet" | Sales | 20 rows |
| 8 | `RETAILER_ID` not unique — 110 rows, 106 unique IDs | Retailer | 4 IDs, 8 rows |
| 9 | Orphan `RETAILER_ID` 999999999 with no matching retailer | Sales | 1 row |

Issue 8 is the most consequential. `RETAILER_ID` is documented as the primary
key of the retailer table, but four IDs appear twice. The identifier encodes
retailer, region, state, and city — so Walmart and West Gear, which share an
initial, collide at locations they both operate. Joining without addressing
this will duplicate sales records and inflate every revenue figure.

## Data Cleaning

Each issue identified above is addressed below, one at a time, with the
reasoning recorded. Cleaning is performed on copies so the raw tables
remain available for comparison.

Rows are removed only when a required value is unusable and cannot be
responsibly inferred. Every removal is counted and reported.

In [9]:
# Work on copies so the raw data stays intact for comparison
sales_clean = sales.copy()
retailers_clean = retailers.copy()

rows_start = len(sales_clean)

In [10]:
# Issue 7: correct the misspelled sales method
sales_clean['SALES_METHOD'] = sales_clean['SALES_METHOD'].replace('Ootlet', 'Outlet')

print(sales_clean['SALES_METHOD'].value_counts())

SALES_METHOD
Online      4889
Outlet      3019
In-store    1740
Name: count, dtype: int64


In [11]:
# Issue 3: convert UNITS_SOLD to numeric; '***' becomes NaN
sales_clean['UNITS_SOLD'] = pd.to_numeric(sales_clean['UNITS_SOLD'], errors='coerce')

# Issue 4: 99999 is a placeholder, not a real price -- median price is $45
sales_clean.loc[sales_clean['PRICE_PER_UNIT'] == 99999, 'PRICE_PER_UNIT'] = np.nan

# Convert the invoice date from text to datetime
sales_clean['INVOICE_DATE'] = pd.to_datetime(sales_clean['INVOICE_DATE'],
                                             format='%m/%d/%Y')

sales_clean[['PRICE_PER_UNIT', 'UNITS_SOLD', 'INVOICE_DATE']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9648 entries, 0 to 9647
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   PRICE_PER_UNIT  9645 non-null   float64       
 1   UNITS_SOLD      9646 non-null   float64       
 2   INVOICE_DATE    9648 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(2)
memory usage: 226.3 KB


In [12]:
# Issues 3, 5, 6: remove rows with unusable price or unit values.
before = len(sales_clean)

sales_clean = sales_clean.dropna(subset=['PRICE_PER_UNIT', 'UNITS_SOLD'])
sales_clean = sales_clean[sales_clean['UNITS_SOLD'] > 0]

print(f"Rows removed: {before - len(sales_clean)} of {before} "
      f"({(before - len(sales_clean)) / before:.2%})")
print(f"Rows remaining: {len(sales_clean):,}")

Rows removed: 9 of 9648 (0.09%)
Rows remaining: 9,639


### Issue 8: Duplicate Retailer IDs

Four `RETAILER_ID` values map to two different retailer records each. In
three cases the conflict is between Walmart and West Gear at the same city;
in one case it is Sports Direct across two different states.

Because `RETAILER_ID` is the only link between a sale and its location,
there is no way to determine from this data which of the two records a
given order belongs to. The affected orders cannot be attributed with
confidence.

We keep one record per ID so the join does not duplicate sales, and we flag
the affected IDs so their impact can be measured. **This is a data
governance issue that should be escalated** — the identifier scheme cannot
distinguish retailers that share an initial and operate in the same city.

In [13]:
# Record which IDs are ambiguous before resolving them
ambiguous_ids = retailers_clean.loc[
    retailers_clean['RETAILER_ID'].duplicated(keep=False), 'RETAILER_ID'
].unique()

# Keep one record per ID so the join cannot duplicate sales rows
retailers_clean = retailers_clean.drop_duplicates(subset='RETAILER_ID', keep='first')

print(f"Retailer rows: {len(retailers)} -> {len(retailers_clean)}")
print(f"IDs now unique: {retailers_clean['RETAILER_ID'].is_unique}")

# Measure how much of the data is affected by the ambiguity
affected = sales_clean[sales_clean['RETAILER_ID'].isin(ambiguous_ids)]
print(f"\nOrders on ambiguous IDs: {len(affected):,} "
      f"({len(affected) / len(sales_clean):.1%} of orders)")
print(f"Units on ambiguous IDs:  {affected['UNITS_SOLD'].sum():,.0f}")

Retailer rows: 110 -> 106
IDs now unique: True

Orders on ambiguous IDs: 623 (6.5% of orders)
Units on ambiguous IDs:  54,248


## Building the Analysis Dataset

With the tables cleaned, we join them into a single dataset. Left joins are
used from the sales table so that any order failing to match is visible
rather than silently dropped.

The data dictionary defines no revenue field, so we derive it as
`PRICE_PER_UNIT × UNITS_SOLD`.

In [14]:
# Join sales to retailer locations and product names
df = (sales_clean
      .merge(retailers_clean, on='RETAILER_ID', how='left')
      .merge(products, on='PRODUCT_ID', how='left'))

# Verify the join did not create or lose rows unexpectedly
print(f"Sales rows in:  {len(sales_clean):,}")
print(f"Rows after join: {len(df):,}")
print(f"Unmatched retailers: {df['RETAILER'].isna().sum()}")
print(f"Unmatched products:  {df['PRODUCT_NAME'].isna().sum()}")

Sales rows in:  9,639
Rows after join: 9,639
Unmatched retailers: 1
Unmatched products:  0


In [15]:
# Issue 9: remove the single order whose RETAILER_ID matches no retailer.
# Without a location this order cannot be attributed to a state or retailer.
df = df.dropna(subset=['RETAILER'])

# Derive revenue -- the source data has no total sales column
df['REVENUE'] = df['PRICE_PER_UNIT'] * df['UNITS_SOLD']

print(f"Final analysis dataset: {len(df):,} rows")
print(f"Date range: {df['INVOICE_DATE'].min():%b %Y} to {df['INVOICE_DATE'].max():%b %Y}")
print(f"Total revenue: ${df['REVENUE'].sum():,.0f}")

Final analysis dataset: 9,638 rows
Date range: Jan 2020 to Dec 2021
Total revenue: $120,045,421
